Count sequences (total seqs, FECV or FIPV seqs) for FIsher's exact test

In [5]:
import pandas as pd
import numpy as np
from Bio import SeqIO
from statsmodels.stats.multitest import multipletests
from scipy.stats import fisher_exact
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.font_manager import FontProperties, findfont

In [3]:
meta = pd.read_csv("/Users/izumikida/FCoV-evolution/Fig2_FIPV_vs_FECV_Fishers_exact_test/Spike/FCoV1_S_all_seq_final_meta_20260306.tsv", sep="\t")
meta

,Accession,biotype,Organism_Name,Isolate,Species,Spike_Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source
0,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,4407,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI
1,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,3699,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI
2,PX315749.1,FIPV,Feline coronavirus,fip_68_liver,Alphacoronavirus suis,3729,United Kingdom,United Kingdom,Felis catus,liver,2013-01-11,2025-09-24,NCBI
3,PX315750.1,FIPV,Feline coronavirus,fip_68_mln,Alphacoronavirus suis,4398,United Kingdom,United Kingdom,Felis catus,lymph,2013-01-11,2025-09-24,NCBI
4,PX315751.1,FIPV,Feline coronavirus,fip_86_liver,Alphacoronavirus suis,4404,United Kingdom,United Kingdom,Felis catus,liver,2013-06-07,2025-09-24,NCBI
...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,JQ304330.1,FECV,Feline coronavirus,FECV141,taxon:12663,348,NaN,Netherlands,Felis catus,feces,NaN,12-JUL-2012,NCBI
444,FJ917532.1,FIPV,Feline coronavirus,FIPV-UCD11b-1a,taxon:12663,186,NaN,USA,Felis catus (cat),diseased lesional tissue,31-Mar-2008,24-JUL-2016,NCBI
445,PP097359.1,FIPV,Feline coronavirus,FIPV/UGM/BUDF,taxon:12663,360,NaN,Indonesia,Felis catus,ascites,NaN,04-FEB-2024,NCBI
446,JQ304361.1,FECV,Feline coronavirus,FECV378,taxon:12663,543,NaN,Netherlands,Felis catus,feces,NaN,12-JUL-2012,NCBI


In [7]:
# load protien alignment
ids = []
seqs = []

for rec in SeqIO.parse("/Users/izumikida/FCoV-evolution/Fig2_FIPV_vs_FECV_Fishers_exact_test/Spike/FCoV1_S_all_seq_cleaned_cds_linsi_curated_fullllength_fix.aa.fasta", "fasta"):
    ids.append(rec.id)
    seqs.append(str(rec.seq))

df_fasta = pd.DataFrame({"ID": ids, "Seq": seqs})
df_fasta

,ID,Seq
0,PQ416775.1,MIVLILALLSTASSEDAPHG-ATLPHFNTSHNNRKFELNFYNFLQT...
1,PX315748.1,MIVLILAFVSVVSSNDAPHG-VTLPHFNNSNN--------------...
2,PX315749.1,MIVLILAFVSVVSSHDAPHG-VTLPQFNTSH---------------...
3,PX315750.1,MIVLILAFVSVVSSHDAPHG-VTLPQFNTSHDNAAFERNFYNFLQT...
4,PX315751.1,MIVLILALLSFARSDDAPHG-VTLPQFNTSHDNAKFELNFYNFLQT...
...,...,...
119,AB088222.1,MILLIFALFSVVRSEDAPHG-VTLPHFNTSYNNPKFELNFYNFLQT...
120,FJ917522.1,MILLLLAILSAVKARDAPHG-VTIPQFNTSHNNAKFELNFYNFLQT...
121,FJ917535.1,MILLLLAILSAVKARDAPHG-VTIPQFNTSHNNAKFELNFYNFLQT...
122,FJ917523.1,MILLLLAILSAVKARDAPHG-VTIPQFNTSHNNAKFELNFYNFLQT...


In [9]:
df_fasta['X_count'] = df_fasta['Seq'].str.count('X')
df_fasta

,ID,Seq,X_count
0,PQ416775.1,MIVLILALLSTASSEDAPHG-ATLPHFNTSHNNRKFELNFYNFLQT...,1
1,PX315748.1,MIVLILAFVSVVSSNDAPHG-VTLPHFNNSNN--------------...,1
2,PX315749.1,MIVLILAFVSVVSSHDAPHG-VTLPQFNTSH---------------...,1
3,PX315750.1,MIVLILAFVSVVSSHDAPHG-VTLPQFNTSHDNAAFERNFYNFLQT...,1
4,PX315751.1,MIVLILALLSFARSDDAPHG-VTLPQFNTSHDNAKFELNFYNFLQT...,1
...,...,...,...
119,AB088222.1,MILLIFALFSVVRSEDAPHG-VTLPHFNTSYNNPKFELNFYNFLQT...,1
120,FJ917522.1,MILLLLAILSAVKARDAPHG-VTIPQFNTSHNNAKFELNFYNFLQT...,1
121,FJ917535.1,MILLLLAILSAVKARDAPHG-VTIPQFNTSHNNAKFELNFYNFLQT...,1
122,FJ917523.1,MILLLLAILSAVKARDAPHG-VTIPQFNTSHNNAKFELNFYNFLQT...,1


In [10]:
df_merged = pd.merge(meta, df_fasta, how="inner", left_on="Accession", right_on="ID")
df_merged

,Accession,biotype,Organism_Name,Isolate,Species,Spike_Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,ID,Seq,X_count
0,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,4407,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,PQ416775.1,MIVLILALLSTASSEDAPHG-ATLPHFNTSHNNRKFELNFYNFLQT...,1
1,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,3699,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,PX315748.1,MIVLILAFVSVVSSNDAPHG-VTLPHFNNSNN--------------...,1
2,PX315749.1,FIPV,Feline coronavirus,fip_68_liver,Alphacoronavirus suis,3729,United Kingdom,United Kingdom,Felis catus,liver,2013-01-11,2025-09-24,NCBI,PX315749.1,MIVLILAFVSVVSSHDAPHG-VTLPQFNTSH---------------...,1
3,PX315750.1,FIPV,Feline coronavirus,fip_68_mln,Alphacoronavirus suis,4398,United Kingdom,United Kingdom,Felis catus,lymph,2013-01-11,2025-09-24,NCBI,PX315750.1,MIVLILAFVSVVSSHDAPHG-VTLPQFNTSHDNAAFERNFYNFLQT...,1
4,PX315751.1,FIPV,Feline coronavirus,fip_86_liver,Alphacoronavirus suis,4404,United Kingdom,United Kingdom,Felis catus,liver,2013-06-07,2025-09-24,NCBI,PX315751.1,MIVLILALLSFARSDDAPHG-VTLPQFNTSHDNAKFELNFYNFLQT...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,AB088222.1,FIPV,Feline infectious peritonitis virus,NaN,taxon:11135,4374,NaN,NaN,NaN,NaN,NaN,26-JUN-2008,NCBI,AB088222.1,MILLIFALFSVVRSEDAPHG-VTLPHFNTSYNNPKFELNFYNFLQT...,1
120,FJ917522.1,FECV,Feline coronavirus,FECV-UCD5,taxon:12663,4413,NaN,USA,Felis catus (cat),diseased lesional tissue,12-Feb-2008,24-JUL-2016,NCBI,FJ917522.1,MILLLLAILSAVKARDAPHG-VTIPQFNTSHNNAKFELNFYNFLQT...,1
121,FJ917535.1,FIPV,Feline coronavirus,FIPV-UCD11b-2b,taxon:12663,4413,NaN,USA,Felis catus (cat),diseased lesional tissue,04-Apr-2008,24-JUL-2016,NCBI,FJ917535.1,MILLLLAILSAVKARDAPHG-VTIPQFNTSHNNAKFELNFYNFLQT...,1
122,FJ917523.1,FIPV,Feline coronavirus,FIPV-UCD13,taxon:12663,4413,NaN,USA,Felis catus (cat),diseased lesional tissue,17-Jan-2008,24-JUL-2016,NCBI,FJ917523.1,MILLLLAILSAVKARDAPHG-VTIPQFNTSHNNAKFELNFYNFLQT...,1


In [ ]:
df_merged['biotype'].value_counts()

biotype
FIPV    74
FECV    50
Name: count, dtype: int64

(B)FCS

In [12]:
# FCS partial seqs were manually obtained from Aliview
# load protien alignment
ids = []
seqs = []

for rec in SeqIO.parse("/Users/izumikida/FCoV-evolution/Fig2_FIPV_vs_FECV_Fishers_exact_test/Spike/FCS_all_aa.fas", "fasta"):
    ids.append(rec.id)
    seqs.append(str(rec.seq))

df_fasta2 = pd.DataFrame({"ID": ids, "Seq": seqs})
df_fasta2

,ID,Seq
0,PQ416775.1,AQAAVINDEIVGAITAVNQTDLFEFVNHTQSRRSRRSTSE----SV...
1,PX315748.1,AQVAVINDEIVGAITAINQTDLFEFVNHTQSRSSRRSTPE----TV...
2,PX315749.1,AQAAVINDEIVGVITSVNQTDLFEFVNHTQPRRARKSIFP----TV...
3,PX315750.1,AQAAVINDEIVGVITSVNQTDLFEFVNHTQPRRARKSIFP----TV...
4,PX315751.1,TQAAVINDEIVGAITAINQTDLFEFVNHTESRRSRMSAPS----TV...
...,...,...
161,HF954968.1,---AVINDEIVGAITAVNQTDLFEFVNHTHLRRSHRSTSE----SV...
162,FJ917534.1,AQAAVINDEIVGVITAVNQTDLFAFVNHTSARKSRRSVTE----SV...
163,HF954950.1,AQAAVINDEIVGVITAVNQTDLFEFVNHTHSKRSRRSTSN----LV...
164,HF954933.1,---AVINDEIVGAITAVNQTDLFEFVNHTQSRRSRRSASN----SV...


In [13]:
df_merged2 = pd.merge(meta, df_fasta2, how="inner", left_on="Accession", right_on="ID")
df_merged2

,Accession,biotype,Organism_Name,Isolate,Species,Spike_Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,ID,Seq
0,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,4407,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,PQ416775.1,AQAAVINDEIVGAITAVNQTDLFEFVNHTQSRRSRRSTSE----SV...
1,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,3699,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,PX315748.1,AQVAVINDEIVGAITAINQTDLFEFVNHTQSRSSRRSTPE----TV...
2,PX315749.1,FIPV,Feline coronavirus,fip_68_liver,Alphacoronavirus suis,3729,United Kingdom,United Kingdom,Felis catus,liver,2013-01-11,2025-09-24,NCBI,PX315749.1,AQAAVINDEIVGVITSVNQTDLFEFVNHTQPRRARKSIFP----TV...
3,PX315750.1,FIPV,Feline coronavirus,fip_68_mln,Alphacoronavirus suis,4398,United Kingdom,United Kingdom,Felis catus,lymph,2013-01-11,2025-09-24,NCBI,PX315750.1,AQAAVINDEIVGVITSVNQTDLFEFVNHTQPRRARKSIFP----TV...
4,PX315751.1,FIPV,Feline coronavirus,fip_86_liver,Alphacoronavirus suis,4404,United Kingdom,United Kingdom,Felis catus,liver,2013-06-07,2025-09-24,NCBI,PX315751.1,TQAAVINDEIVGAITAINQTDLFEFVNHTESRRSRMSAPS----TV...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161,HF954968.1,FIPV,Feline coronavirus,FIPV D04 93-1,taxon:12663,147,NaN,USA:PA,Felis catus,kidney of a diseased cat,2004,03-JUL-2013,NCBI,HF954968.1,---AVINDEIVGAITAVNQTDLFEFVNHTHLRRSHRSTSE----SV...
162,FJ917534.1,FIPV,Feline coronavirus,FIPV-UCD11b-2a,taxon:12663,4413,NaN,USA,Felis catus (cat),diseased lesional tissue,04-Apr-2008,24-JUL-2016,NCBI,FJ917534.1,AQAAVINDEIVGVITAVNQTDLFAFVNHTSARKSRRSVTE----SV...
163,HF954950.1,FECV,Feline coronavirus,FECV 167,taxon:12663,156,NaN,USA:RI,Felis catus,feces of a healthy cat,2010,03-JUL-2013,NCBI,HF954950.1,AQAAVINDEIVGVITAVNQTDLFEFVNHTHSKRSRRSTSN----LV...
164,HF954933.1,FECV,Feline coronavirus,FECV 131,taxon:12663,148,NaN,USA:WI,Felis catus,feces of a healthy cat,2010,03-JUL-2013,NCBI,HF954933.1,---AVINDEIVGAITAVNQTDLFEFVNHTQSRRSRRSASN----SV...


In [14]:
df_merged2['biotype'].value_counts()

biotype
FIPV    94
FECV    72
Name: count, dtype: int64

(C) M1058L region

In [16]:
# FCS partial seqs were manually obtained from Aliview
# load protien alignment
ids = []
seqs = []

for rec in SeqIO.parse("/Users/izumikida/FCoV-evolution/Fig2_FIPV_vs_FECV_Fishers_exact_test/Spike/M1058_all_seq.aa.fasta", "fasta"):
    ids.append(rec.id)
    seqs.append(str(rec.seq))

df_fasta3 = pd.DataFrame({"ID": ids, "Seq": seqs})
df_fasta3

,ID,Seq
0,PQ416775.1,VCAQYYNGIMVLPGVVDGNKMAMYTASLIGGMALGSITSAVAVPFA...
1,PX315748.1,VCAQYYNGIMVLPGVVDQNKMAMYTASLIGGMAMGAITSAVAVPFA...
2,PX315749.1,VCAQYYNGIMVLPGVVDANKMAMYTASLIGGMALGSITSAVAVPFA...
3,PX315750.1,VCAQYYNGIMVLPGVVDANKMAMYTASLIGGMALGSITSAVAVPFA...
4,PX315751.1,VCAQYYNGIMVLPGVVDQNKMAMYTASLIGGMALGSITSAVAVPFA...
...,...,...
362,JQ304388.1,-----------------------YTASLIGGMAMGSITSAVAVPFA...
363,JQ304500.1,---------------------------LIGGMALGSITSAVAVPFA...
364,JQ304330.1,---------------------AMYTASLIGGMAMGSITSTVAVPFA...
365,JQ304361.1,---------------------------LIGGMAMGSITSAVAVPFA...


In [17]:
df_merged3 = pd.merge(meta, df_fasta3, how="inner", left_on="Accession", right_on="ID")
df_merged3

,Accession,biotype,Organism_Name,Isolate,Species,Spike_Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Collection_Date,Release_Date,Source,ID,Seq
0,PQ416775.1,FIPV,Feline coronavirus,H30,Alphacoronavirus suis,4407,China: Shandong,China,Felis catus,liver,2023-11-05,2025-10-01,NCBI,PQ416775.1,VCAQYYNGIMVLPGVVDGNKMAMYTASLIGGMALGSITSAVAVPFA...
1,PX315748.1,FIPV,Feline coronavirus,fip_37_liver,Alphacoronavirus suis,3699,United Kingdom,United Kingdom,Felis catus,liver,2011-10,2025-09-24,NCBI,PX315748.1,VCAQYYNGIMVLPGVVDQNKMAMYTASLIGGMAMGAITSAVAVPFA...
2,PX315749.1,FIPV,Feline coronavirus,fip_68_liver,Alphacoronavirus suis,3729,United Kingdom,United Kingdom,Felis catus,liver,2013-01-11,2025-09-24,NCBI,PX315749.1,VCAQYYNGIMVLPGVVDANKMAMYTASLIGGMALGSITSAVAVPFA...
3,PX315750.1,FIPV,Feline coronavirus,fip_68_mln,Alphacoronavirus suis,4398,United Kingdom,United Kingdom,Felis catus,lymph,2013-01-11,2025-09-24,NCBI,PX315750.1,VCAQYYNGIMVLPGVVDANKMAMYTASLIGGMALGSITSAVAVPFA...
4,PX315751.1,FIPV,Feline coronavirus,fip_86_liver,Alphacoronavirus suis,4404,United Kingdom,United Kingdom,Felis catus,liver,2013-06-07,2025-09-24,NCBI,PX315751.1,VCAQYYNGIMVLPGVVDQNKMAMYTASLIGGMALGSITSAVAVPFA...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
362,JQ304388.1,FECV,Feline coronavirus,FECV577,taxon:12663,461,NaN,Netherlands,Felis catus,feces,NaN,12-JUL-2012,NCBI,JQ304388.1,-----------------------YTASLIGGMAMGSITSAVAVPFA...
363,JQ304500.1,FIPV,Feline coronavirus,FIPV826,taxon:12663,391,NaN,Netherlands,Felis catus,internal organ,NaN,12-JUL-2012,NCBI,JQ304500.1,---------------------------LIGGMALGSITSAVAVPFA...
364,JQ304330.1,FECV,Feline coronavirus,FECV141,taxon:12663,348,NaN,Netherlands,Felis catus,feces,NaN,12-JUL-2012,NCBI,JQ304330.1,---------------------AMYTASLIGGMAMGSITSTVAVPFA...
365,JQ304361.1,FECV,Feline coronavirus,FECV378,taxon:12663,543,NaN,Netherlands,Felis catus,feces,NaN,12-JUL-2012,NCBI,JQ304361.1,---------------------------LIGGMAMGSITSAVAVPFA...


In [18]:
df_merged3['biotype'].value_counts()

biotype
FIPV    211
FECV    156
Name: count, dtype: int64